# WP4v3 — Notebook 3 : Inférence

**Normalisation cohérente à l'inférence** :
- CLS tokens → normalisés avec `cls_norm_stats` (mêmes stats qu'à l'entraînement)
- Patch tokens → normalisés avec `patch_norm_stats` (distribution différente)

**Pipeline** :
```
Image → MAE → patch tokens (196, 1024)
      → normalisation (patch_norm_stats)
      → f_theta (point-wise) → (196, 1024) espace CLIP
      → MLP connector LLaVA → (196, 4096) espace LLM
      → LLM (template Vicuna) → description
```

## 1. Chargement des modèles et stats

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import ViTMAEModel, ViTImageProcessor
from transformers import LlavaForConditionalGeneration, CLIPImageProcessor, LlamaTokenizer
from datasets import load_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

# f_theta
class ProjectionMLP(nn.Module):
    def __init__(self, dim=1024, hidden_dim=2048, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, dim),
        )
    def forward(self, x): return F.normalize(self.net(x), dim=-1)

f_theta = ProjectionMLP().to(DEVICE)
f_theta.load_state_dict(torch.load('wp4v3_ftheta_best.pt'))
f_theta.eval()

# Stats de normalisation
cls_norm   = torch.load('wp4v3_cls_norm_stats.pt')
patch_norm = torch.load('wp4v3_patch_norm_stats.pt')
cls_mean, cls_std     = cls_norm['mean'].cpu(),   cls_norm['std'].cpu()
patch_mean, patch_std = patch_norm['mean'].cpu(), patch_norm['std'].cpu()

print('Stats chargees')
print(f'  CLS   mean norm: {cls_mean.norm():.3f}  | std norm: {cls_std.norm():.3f}')
print(f'  Patch mean norm: {patch_mean.norm():.3f} | std norm: {patch_std.norm():.3f}')


In [ ]:
# MAE
mae_processor = ViTImageProcessor(
    size={'height': 224, 'width': 224},
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)
mae_encoder = ViTMAEModel.from_pretrained('./vit-mae-large').to(DEVICE)
mae_encoder.eval()

# LLaVA complet
llava_full = LlavaForConditionalGeneration.from_pretrained(
    './llava-1.5-7b-hf', torch_dtype=torch.float16
).to(DEVICE)
llava_full.eval()
llm          = llava_full.language_model
vision_tower = llava_full.vision_tower
mlp_conn     = llava_full.multi_modal_projector
clip_proc    = CLIPImageProcessor.from_pretrained('./llava-1.5-7b-hf')
tokenizer    = LlamaTokenizer.from_pretrained('./llava-1.5-7b-hf', use_fast=False)

print(f'VRAM : {torch.cuda.memory_allocated()/1e9:.1f} GB')


## 2. Fonctions utilitaires

In [ ]:
SYSTEM    = ('A chat between a curious user and an artificial intelligence assistant. '
             'The assistant gives helpful, detailed, and polite answers to the user questions.')
USER_TEXT = 'Describe this image in one sentence.'
before_ids    = tokenizer(f'{SYSTEM} USER: ', return_tensors='pt', add_special_tokens=True).input_ids.to(DEVICE)
after_ids     = tokenizer(f'\n{USER_TEXT} ASSISTANT:', return_tensors='pt', add_special_tokens=False).input_ids.to(DEVICE)
before_embeds = llm.get_input_embeddings()(before_ids).half()
after_embeds  = llm.get_input_embeddings()(after_ids).half()


def project_with_cls_norm(token):
    """Pour les CLS tokens — memes stats qu'a l'entrainement."""
    z_norm = (token.cpu().float() - cls_mean) / cls_std
    with torch.no_grad():
        return f_theta(z_norm.unsqueeze(0).to(DEVICE))[0].cpu().float()


def project_with_patch_norm(tokens):
    """
    Pour les patch tokens (196 ou 49) — stats patch.
    tokens : Tensor (N, 1024)
    Retourne : Tensor (N, 1024) L2-normalise dans espace CLIP
    """
    z_norm = (tokens.cpu().float() - patch_mean) / patch_std  # (N, 1024)
    with torch.no_grad():
        return f_theta(z_norm.to(DEVICE)).cpu().float()        # (N, 1024)


def llm_describe(visual_tokens_clip):
    """tokens CLIP (N, 1024) -> MLP connector -> LLM -> description."""
    with torch.no_grad():
        visual_llm = mlp_conn(visual_tokens_clip.half().to(DEVICE))  # (N, 4096)
    inputs_embeds = torch.cat([
        before_embeds, visual_llm.unsqueeze(0), after_embeds
    ], dim=1)
    with torch.no_grad():
        out_ids = llm.generate(
            inputs_embeds=inputs_embeds, max_new_tokens=50,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()


def get_mae_tokens(image_pil, seed=None):
    """Retourne CLS + patch tokens MAE. seed=None -> full encoding."""
    inp = mae_processor(images=image_pil, return_tensors='pt')
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    if seed is None:
        noise = torch.zeros(1, 196).to(DEVICE)
    else:
        gen   = torch.Generator().manual_seed(seed)
        noise = torch.rand(1, 196, generator=gen).to(DEVICE)
    with torch.no_grad():
        out = mae_encoder(**inp, noise=noise)
    cls_token    = out.last_hidden_state[0, 0].cpu().float()   # (1024,)
    patch_tokens = out.last_hidden_state[0, 1:].cpu().float()  # (196 ou 49, 1024)
    mask         = out.mask[0].cpu()
    visible_ids  = torch.where(mask == 0)[0].tolist()
    return cls_token, patch_tokens, visible_ids


def describe_llava_native(image_pil):
    """LLaVA natif : 576 patch tokens CLIP -> MLP connector -> LLM."""
    inp = clip_proc(images=image_pil, return_tensors='pt', do_rescale=True)
    pix = inp['pixel_values'].to(DEVICE).half()
    with torch.no_grad():
        vis     = vision_tower(pix).last_hidden_state[:, 1:]  # (1, 576, 1024)
        vis_llm = mlp_conn(vis)[0]                            # (576, 4096)
    inputs_embeds = torch.cat([before_embeds, vis_llm.unsqueeze(0), after_embeds], dim=1)
    with torch.no_grad():
        out_ids = llm.generate(
            inputs_embeds=inputs_embeds, max_new_tokens=50,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()


print('Fonctions definies')


## 3. Vérification discriminabilité après projection

In [ ]:
ds = load_dataset('parquet', data_files={
    'validation': './imagenet100/data/validation-*.parquet',
})

raw_vecs, proj_vecs = [], []
for i in range(5):
    item      = ds['validation'][i]
    image_pil = item['image'].convert('RGB').resize((224, 224))
    cls_token, _, _ = get_mae_tokens(image_pil, seed=None)
    z_proj = project_with_cls_norm(cls_token)  # normalise avec cls_norm_stats
    raw_vecs.append(F.normalize(cls_token.unsqueeze(0), dim=-1)[0])
    proj_vecs.append(z_proj)
    print(f'{item["text"][:38]}')

raw_mat  = torch.stack(raw_vecs)
proj_mat = torch.stack(proj_vecs)
print('\nSimilarites CLS MAE brut :')
print((raw_mat @ raw_mat.T).numpy().round(4))
print('\nSimilarites apres projection f_theta :')
print((proj_mat @ proj_mat.T).numpy().round(4))


## 4. Sanity check — LLaVA natif vs MAE projeté (196 patch tokens)

In [ ]:
N_IMAGES = 5
print(f'{"Classe":<38} {"LLaVA natif (576 patches)":<45} {"MAE -> f_theta -> MLP (196 patches)"}')
print('-' * 125)

for i in range(N_IMAGES):
    item      = ds['validation'][i]
    image_pil = item['image'].convert('RGB')
    label     = item['text'][:35]

    # LLaVA natif
    desc_llava = describe_llava_native(image_pil)

    # MAE : 196 patch tokens -> normalisation patch_norm -> f_theta -> MLP connector -> LLM
    image_224 = image_pil.resize((224, 224))
    _, patch_tokens, _ = get_mae_tokens(image_224, seed=None)    # (196, 1024) full encoding
    clip_space = project_with_patch_norm(patch_tokens)            # (196, 1024) normalise avec patch_norm
    desc_mae   = llm_describe(clip_space)                         # MLP connector -> LLM

    print(f'{label:<38} {desc_llava:<45} {desc_mae}')


## 5. Test avec masquage — 49 patches visibles

In [ ]:
IMG_IDX = 0; SEED = 42
item      = ds['validation'][IMG_IDX]
image_pil = item['image'].convert('RGB').resize((224, 224))
label_txt = item['text']

cls_token, patch_tokens, visible_ids = get_mae_tokens(image_pil, seed=SEED)
print(f'Classe : {label_txt} | Patches visibles : {len(visible_ids)}')

# Description depuis 49 patch tokens masques
clip_masked = project_with_patch_norm(patch_tokens)  # (49, 1024)
desc_masked = llm_describe(clip_masked)
print(f'Description (49 patches masques) : {desc_masked}')

# Description depuis 196 patch tokens full
_, full_tokens, _ = get_mae_tokens(image_pil, seed=None)
clip_full    = project_with_patch_norm(full_tokens)  # (196, 1024)
desc_full    = llm_describe(clip_full)
print(f'Description (196 patches full)   : {desc_full}')

# Visualisation masque
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
overlay = np.array(image_pil).copy().astype(float)
for pid in range(196):
    if pid not in visible_ids:
        r, c = pid // 14, pid % 14
        overlay[r*16:(r+1)*16, c*16:(c+1)*16] *= 0.15
ax.imshow(overlay.astype(np.uint8))
ax.set_title(label_txt, fontsize=9); ax.axis('off')
plt.tight_layout()
plt.savefig('wp4v3_image_masque.png', dpi=150, bbox_inches='tight')
plt.show()
